---
title: "In Class Assingment 2"
author: Karisa Kopecek
date: today
format:
  html:
    embed-resources: true
    echo: true
---

## Load the credit card dataset and perform some initial EDA. In a markdown cell discuss some highlights from your EDA.

In [1]:
#importing libraries
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
#from sklearn.preprocessing import MinMaxScaler skipped for today b/c tree based
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score

In [2]:
# importing data
df = pd.read_csv('creditcard.csv')

# renaming target for easier referencing
df = df.rename(columns={'default payment next month': 'default'})

df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,400000,1,1,2,32,0,0,0,0,0,0,55773,55917,51389,48272,49478,51242,3028,3023,3000,3000,3000,38662,0
1,120000,2,2,2,30,-1,-1,-1,-1,-1,-1,140,3230,3011,1964,1883,1538,3230,3011,1964,1883,1538,1911,0
2,270000,2,2,2,32,0,0,0,0,0,0,59710,49986,104390,94856,86461,83650,1808,69563,2891,2689,3012,2771,0
3,280000,2,2,1,27,0,0,0,0,0,0,280913,283222,273160,257689,193231,191143,11052,9563,15017,5374,5420,6021,0
4,30000,2,1,2,27,0,0,-1,0,0,-2,1512,2458,664,1814,0,0,1000,664,1500,0,0,0,0


In [3]:
# loading the dataset and initial EDA with SweetViz
creditcard = pd.read_csv('creditcard.csv')
creditcard.head()

# visualizing the data with SweetViz
report = sv.analyze(creditcard)
report.show_html('creditcard.html')

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report creditcard.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


Highlights of EDA:
This dataset contains 24,000 records with no missing values and includes details on customer demographics, credit limits, monthly bills, payments, and repayment history. Bill amounts and payment amounts are heavily concentrated near zero with some large outliers, showing that many customers carry little balance while a smaller group carries much higher debt. Most customers are between about 30 and 40 years old, and education is dominated by a few groups. There is an even split of married people vs not married people in the data. Repayment status values are often around zero, meaning many customers pay on time, and behavior tends to be consistent across the months. More suctomers didn't default, so there is also a class imbalance in the data. 

## Prepare the data and build default tuned RandomForestClassifier and XGBClassifier models (use the AI-generated defaults for now). Do this with both the entire data set and using 5-fold cross-validation.  Calculate the mean metric score and standard deviation for the folds. In a markdown cell briefly discuss how your models performed. What does the mean and standard deviation across the folds tell you?

In [4]:
# split features and target
X = df.drop('default', axis=1)
y = df['default']

# train/test split - stratified to preserve class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")
print(f"Train default rate: {y_train.mean():.1%}")
print(f"Test default rate: {y_test.mean():.1%}")

Training set: 19200 rows
Test set: 4800 rows
Train default rate: 22.1%
Test default rate: 22.1%


In [5]:
# build default RF and XGB models on the full train/test split
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("=== Default Random Forest ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"F1 (macro): {f1_score(y_test, y_pred_rf, average='macro'):.4f}")

print("\n=== Default XGBoost ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"F1 (macro): {f1_score(y_test, y_pred_xgb, average='macro'):.4f}")

=== Default Random Forest ===
Accuracy: 0.8115
F1 (macro): 0.6770

=== Default XGBoost ===
Accuracy: 0.8102
F1 (macro): 0.6778


In [6]:
# 5-fold cross validation - using f1_macro as the scoring metric
# f1 is more informative than accuracy for imbalanced classes
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_cv = cross_val_score(rf, X, y, cv=cv, scoring='f1_macro')
xgb_cv = cross_val_score(xgb, X, y, cv=cv, scoring='f1_macro')

print("=== 5-Fold Cross Validation (F1 Macro) ===")
print(f"\nRandom Forest fold scores: {np.round(rf_cv, 4)}")
print(f"  Mean: {rf_cv.mean():.4f}  |  Std: {rf_cv.std():.4f}")

print(f"\nXGBoost fold scores:       {np.round(xgb_cv, 4)}")
print(f"  Mean: {xgb_cv.mean():.4f}  |  Std: {xgb_cv.std():.4f}")

=== 5-Fold Cross Validation (F1 Macro) ===

Random Forest fold scores: [0.6784 0.674  0.6786 0.6771 0.6705]
  Mean: 0.6757  |  Std: 0.0031

XGBoost fold scores:       [0.6761 0.6761 0.6734 0.6858 0.6729]
  Mean: 0.6768  |  Std: 0.0047


**Calculate the mean metric score and standard deviation for the folds.**
RF had a mean macro F1 of 0.6757 with a standard deviation of 0.0031. XGBoost had a mean macro F1 of 0.6768 with a standard deviation of 0.0047.

**Briefly discuss how your models performed.**
Both models performed similarly and kind of generally ok, not very good (but not extremely bad either). RF hit 81.15% accuracy and XGBoost 81.02%, with almost the same macro F1 scores (0.6770 and 0.6778). The dataset is 78% non-defaulters, meaning a model that always predicts no default would score well on accuracy without being useful (0.677 is mediocre). So, it is doing well on accuracy but I don't think it is preforming well on the minority class. 

**What does the mean and standard deviation across the folds tell you?**
The low standard deviations (0.0031 for RF, 0.0047 for XGBoost) tell us both models are stable and consistent across folds (neither is overfitting to a particular data split as discussed in class). The means are close enough that there is no meaningful difference between the two models at default settings.

## Look at the classification report for your two models. Does this change your evaluation of the models?


In [7]:
print("=== Random Forest Classification Report ===")
print(classification_report(y_test, y_pred_rf, target_names=['No Default', 'Default']))

print("=== XGBoost Classification Report ===")
print(classification_report(y_test, y_pred_xgb, target_names=['No Default', 'Default']))

=== Random Forest Classification Report ===
              precision    recall  f1-score   support

  No Default       0.84      0.94      0.89      3738
     Default       0.62      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800

=== XGBoost Classification Report ===
              precision    recall  f1-score   support

  No Default       0.84      0.93      0.88      3738
     Default       0.61      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800



Yes, the classification reports change the evaluation significantly. Both models recall only 38% of actual defaulters, meaning they miss 62% of customers who will default. Precision on the default class is 0.62 for RF and 0.61 for XGB, which is passable, but precision doesn't matter much if the model isn't flagging defaulters in the first place. The not defaulting class tells the opposite story with recall of 0.94 (RF) and 0.93 (XGB), which is why overall accuracy looks fine at 81%. The models are very good at identifying safe customers and poor at identifying risky ones. My intuition earlier was correct about the model accuracy score not being a good representative because it was overstating how useful for models were. 

## Build your models using cross validation again, but this time for both use model features to adjust for the class imbalance. How did this impact model performance?


In [8]:
# RF: class_weight='balanced' adjusts weights inversely proportional to class frequency
rf_balanced = RandomForestClassifier(random_state=42, class_weight='balanced')

# XGB: scale_pos_weight = ratio of negative to positive class samples
scale = (y == 0).sum() / (y == 1).sum()
print(f"XGBoost scale_pos_weight: {scale:.2f}")
xgb_balanced = XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=scale)

# fit on train set
rf_balanced.fit(X_train, y_train)
xgb_balanced.fit(X_train, y_train)

y_pred_rf_bal = rf_balanced.predict(X_test)
y_pred_xgb_bal = xgb_balanced.predict(X_test)

print("\n=== Random Forest (Balanced) ===")
print(classification_report(y_test, y_pred_rf_bal, target_names=['No Default', 'Default']))

print("=== XGBoost (Balanced) ===")
print(classification_report(y_test, y_pred_xgb_bal, target_names=['No Default', 'Default']))

XGBoost scale_pos_weight: 3.52

=== Random Forest (Balanced) ===
              precision    recall  f1-score   support

  No Default       0.84      0.94      0.89      3738
     Default       0.65      0.36      0.46      1062

    accuracy                           0.82      4800
   macro avg       0.74      0.65      0.68      4800
weighted avg       0.80      0.82      0.79      4800

=== XGBoost (Balanced) ===
              precision    recall  f1-score   support

  No Default       0.87      0.82      0.84      3738
     Default       0.47      0.58      0.52      1062

    accuracy                           0.76      4800
   macro avg       0.67      0.70      0.68      4800
weighted avg       0.79      0.76      0.77      4800



In [9]:
# 5-fold CV on balanced models
rf_bal_cv = cross_val_score(rf_balanced, X, y, cv=cv, scoring='f1_macro')
xgb_bal_cv = cross_val_score(xgb_balanced, X, y, cv=cv, scoring='f1_macro')

print("=== Balanced Models - 5-Fold CV (F1 Macro) ===")
print(f"\nRF (Balanced) - Mean: {rf_bal_cv.mean():.4f}  |  Std: {rf_bal_cv.std():.4f}")
print(f"XGB (Balanced) - Mean: {xgb_bal_cv.mean():.4f}  |  Std: {xgb_bal_cv.std():.4f}")

print("\n=== Comparison to Default Models ===")
print(f"RF default F1:   {rf_cv.mean():.4f}  -> Balanced: {rf_bal_cv.mean():.4f}")
print(f"XGB default F1:  {xgb_cv.mean():.4f}  -> Balanced: {xgb_bal_cv.mean():.4f}")

=== Balanced Models - 5-Fold CV (F1 Macro) ===

RF (Balanced) - Mean: 0.6682  |  Std: 0.0037
XGB (Balanced) - Mean: 0.6772  |  Std: 0.0064

=== Comparison to Default Models ===
RF default F1:   0.6757  -> Balanced: 0.6682
XGB default F1:  0.6768  -> Balanced: 0.6772


The impact differed significantly between the two models. RF with class_weight='balanced' actually got slightly worse. Default recall dropped from 0.38 to 0.36 and macro F1 fell from 0.6757 to 0.6682, so the adjustment did not help RF at all. XGBoost responded much better. Setting scale_pos_weight=3.52 pushed default recall from 0.38 to 0.58, meaning the model now catches over half of actual defaulters instead of just over a third. The tradeoff is lower precision on the default class (0.61 down to 0.47), so there are more false alarms, and overall accuracy dropped to 0.76. The macro F1 stayed nearly flat at 0.6772, but the shift toward higher recall is more useful for the overall goal of catching a defaulter. Overall, class balancing is worth applying to XGBoost but not to RF based on these results or maybe I would need to spend more time outside class doing further testing (but this was an in-class assignment).

## Now try the XGBoost boostrapping (subsample =0.8) feature with. How did this affect performance?


In [10]:
# subsample=0.8 means each tree is trained on a random 80% of training rows
# this introduces variance reduction similar to bagging and can help reduce overfitting
xgb_subsample = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=scale,
    subsample=0.8
)
xgb_subsample.fit(X_train, y_train)
y_pred_xgb_sub = xgb_subsample.predict(X_test)

# CV scores
xgb_sub_cv = cross_val_score(xgb_subsample, X, y, cv=cv, scoring='f1_macro')

print("=== XGBoost with subsample=0.8 ===")
print(classification_report(y_test, y_pred_xgb_sub, target_names=['No Default', 'Default']))
print(f"CV Mean F1: {xgb_sub_cv.mean():.4f}  |  Std: {xgb_sub_cv.std():.4f}")
print(f"\nComparison - Balanced XGB (no subsample): {xgb_bal_cv.mean():.4f}")
print(f"Balanced XGB with subsample=0.8:          {xgb_sub_cv.mean():.4f}")

=== XGBoost with subsample=0.8 ===
              precision    recall  f1-score   support

  No Default       0.87      0.81      0.84      3738
     Default       0.46      0.57      0.51      1062

    accuracy                           0.76      4800
   macro avg       0.67      0.69      0.68      4800
weighted avg       0.78      0.76      0.77      4800

CV Mean F1: 0.6687  |  Std: 0.0040

Comparison - Balanced XGB (no subsample): 0.6772
Balanced XGB with subsample=0.8:          0.6687


Adding subsample=0.8 trains each tree on a random 80% of training rows to reduce overfitting, but it did not help here. CV mean F1 dropped from 0.6772 to 0.6687 compared to the balanced XGBoost, and the classification report was mostly the same. Since the model was not overfitting to begin with, subsampling had little to offer plus as discussed in class XGBoost tuning can be very complex and sometimes can't be handled in a single class period. 

## Do a brief tuning (2 or 3 features) for each model. It is your choice about whether to use the class imbalance adjustments or the subsample feature. Did your model performance increase or decrease? Do you think you chose the right parameters to tune?


In [11]:
# Random Forest tuning
# Tuning min_samples_leaf and max_features (different from the class example which used n_estimators and max_depth, did this on purpose to try to see something new)
# min_samples_leaf- definition from online- controls leaf node size, higher values = more regularization
# max_features- controls feature subset size at each split

rf_param_grid = {
    'min_samples_leaf': [1, 4, 8],
    'max_features': ['sqrt', 'log2']
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid=rf_param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)

print("Best RF Parameters:", rf_grid.best_params_)
print(f"Best RF CV F1 (macro): {rf_grid.best_score_:.4f}")

best_rf = rf_grid.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)
print("\nBest RF Classification Report:")
print(classification_report(y_test, y_pred_best_rf, target_names=['No Default', 'Default']))

Best RF Parameters: {'max_features': 'sqrt', 'min_samples_leaf': 8}
Best RF CV F1 (macro): 0.7057

Best RF Classification Report:
              precision    recall  f1-score   support

  No Default       0.87      0.86      0.87      3738
     Default       0.53      0.55      0.54      1062

    accuracy                           0.79      4800
   macro avg       0.70      0.71      0.70      4800
weighted avg       0.80      0.79      0.79      4800



In [12]:
# XGBoost tuning
# Tuning learning_rate and min_child_weight (different from the class example which used n_estimators and max_depth)
# learning_rate: step size shrinkage to prevent overfitting
# min_child_weight: minimum sum of instance weights in a leaf, higher = more conservative

xgb_param_grid = {
    'learning_rate': [0.05, 0.1, 0.2],
    'min_child_weight': [1, 5, 10]
}

xgb_grid = GridSearchCV(
    estimator=XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=scale, subsample=0.8),
    param_grid=xgb_param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)
xgb_grid.fit(X_train, y_train)

print("Best XGB Parameters:", xgb_grid.best_params_)
print(f"Best XGB CV F1 (macro): {xgb_grid.best_score_:.4f}")

best_xgb = xgb_grid.best_estimator_
y_pred_best_xgb = best_xgb.predict(X_test)
print("\nBest XGB Classification Report:")
print(classification_report(y_test, y_pred_best_xgb, target_names=['No Default', 'Default']))

Best XGB Parameters: {'learning_rate': 0.05, 'min_child_weight': 5}
Best XGB CV F1 (macro): 0.6914

Best XGB Classification Report:
              precision    recall  f1-score   support

  No Default       0.88      0.80      0.84      3738
     Default       0.47      0.62      0.54      1062

    accuracy                           0.76      4800
   macro avg       0.68      0.71      0.69      4800
weighted avg       0.79      0.76      0.77      4800



In [13]:
# summary comparison across all model versions
#had AI make me a summery table to see things more easily
print("=" * 55)
print("FULL MODEL COMPARISON SUMMARY")
print("=" * 55)
results = {
    'RF Default':          f1_score(y_test, y_pred_rf, average='macro'),
    'XGB Default':         f1_score(y_test, y_pred_xgb, average='macro'),
    'RF Balanced':         f1_score(y_test, y_pred_rf_bal, average='macro'),
    'XGB Balanced':        f1_score(y_test, y_pred_xgb_bal, average='macro'),
    'XGB Subsample=0.8':   f1_score(y_test, y_pred_xgb_sub, average='macro'),
    'RF Tuned':            f1_score(y_test, y_pred_best_rf, average='macro'),
    'XGB Tuned':           f1_score(y_test, y_pred_best_xgb, average='macro'),
}

for name, score in results.items():
    print(f"  {name:<25} F1 Macro: {score:.4f}")

FULL MODEL COMPARISON SUMMARY
  RF Default                F1 Macro: 0.6770
  XGB Default               F1 Macro: 0.6778
  RF Balanced               F1 Macro: 0.6758
  XGB Balanced              F1 Macro: 0.6836
  XGB Subsample=0.8         F1 Macro: 0.6751
  RF Tuned                  F1 Macro: 0.7035
  XGB Tuned                 F1 Macro: 0.6879


**Random Forest tuning (min_samples_leaf, max_features):** The best parameters were min_samples_leaf=8 and max_features='sqrt'. Increasing min_samples_leaf forces larger leaf nodes, which reduces overfitting by preventing trees from fitting to noise. The tuned RF reached a test F1 of 0.7035, up from 0.6770 at default.

**XGBoost tuning (learning_rate, min_child_weight):** The best parameters were learning_rate=0.05 and min_child_weight=5. The tuned XGB hit a test F1 of 0.6879, up from 0.6778 at default.

**Did your model performance increase or decrease?** Both models improved. RF went from 0.6770 to 0.7035 and XGBoost went from 0.6778 to 0.6879.

**Did you choose the right parameters to tune?** Yes, particularly for RF where min_samples_leaf had a clear impact. For XGBoost the gain was smaller, so exploring n_estimators or max_depth alongside learning_rate might have produced a bigger improvement.

## Which of your models was better out-of-the-box? Be specific.

Both models were nearly identical out of the box. RF scored 0.6770 macro F1 and XGBoost scored 0.6778, a difference of less than 0.001. If forced to pick, XGBoost was technically better out of the box by a very small margin. However, RF ended up being the better model overall after tuning, reaching a macro F1 of 0.7035 compared to XGBoost's 0.6879. This findng agrees with what was stated in class about XGBoost being better out of the box but harder to tune. 